# Task 2 — Generative AI Domain-Specific Fine-Tuning Pipeline
**Use case: AI Code Review Assistant**


In [5]:
!pip install -q -U "pip<25.3" "setuptools<82" wheel

!pip install -q -U \
    "transformers==4.57.6" \
    "tokenizers==0.22.2" \
    "peft==0.20.0" \
    "bitsandbytes==0.50.2" \
    "trl==0.29.1" \
    "huggingface-hub>=0.34,<1.0" \
    "accelerate>=0.33" \
    "datasets>=2.20" \
    "groq" \
    "rouge-score" \
    "bert-score" \
    "chromadb" \
    "sentence-transformers"

print("Install complete. If this is the first install this session, "
      "Runtime → Restart session, then re-run from the top.")

Install complete. If this is the first install this session, Runtime → Restart session, then re-run from the top.


In [1]:
import os
import json
import random
import re
import time
import textwrap
from datetime import datetime

from google.colab import userdata

random.seed(42)

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

os.makedirs("task2_genai/data", exist_ok=True)
os.makedirs("task2_genai/logs", exist_ok=True)
os.makedirs("task2_genai/model", exist_ok=True)

print("Setup complete.")

Setup complete.


## Task 2A — Use Case Definition and Dataset Engineering

### Problem statement

| | |
|---|---|
| **Input** | A short Python function/snippet (10-60 lines), optionally with a one-line context of what it's supposed to do |
| **Output** | A structured JSON review: a list of `issues`, each with `category` (bug / security / performance / style / readability), `severity` (critical / major / minor), `line_hint`, and `suggestion`; plus an `overall_verdict` (approve / request_changes) and `summary` |
| **Correct response** | Every real issue in the snippet is caught, categorised and severity-ranked correctly, with an actionable (not generic) suggestion; verdict is `request_changes` iff at least one `critical`/`major` issue exists |
| **Incorrect response** | Misses a planted bug, invents an issue that isn't there (hallucination), wrong severity/verdict logic, or a suggestion that's too vague to act on (e.g. "improve this") |


In [2]:

TEACHER_SYSTEM_PROMPT = """You are a principal-level Python code reviewer generating TRAINING DATA
for fine-tuning a smaller model to do automated code review.

You will be given several scenarios at once, numbered. For EACH scenario you must:
1. Write a realistic Python function (15-45 lines) that plausibly appears in production code,
   containing 1-3 DELIBERATE, REALISTIC issues drawn from the requested category mix
   (bugs, security flaws, performance problems, style/readability issues). Vary difficulty and domain
   (web backends, data pipelines, CLI tools, ML preprocessing, financial calculations, etc.)
2. Produce the ground-truth review as STRICT JSON matching this schema exactly, nothing else:

{
  "issues": [
    {"category": "bug|security|performance|style|readability",
     "severity": "critical|major|minor",
     "line_hint": "<short quote or line description>",
     "suggestion": "<specific, actionable fix, 1-2 sentences>"}
  ],
  "overall_verdict": "approve|request_changes",
  "summary": "<2-3 sentence summary>"
}

Respond with a single JSON object of the form:
{"examples": [{"code": "...", "review": {...}}, {"code": "...", "review": {...}}, ...]}
— exactly one entry per numbered scenario, in the same order given.
No markdown fences, no commentary outside the JSON."""

# Scenario pool ensures topical + length diversity
SCENARIO_SEEDS = [
    "a Flask endpoint handling file uploads",
    "a pandas function cleaning a financial transactions dataframe",
    "a function computing compound interest",
    "a retry wrapper for an external API call",
    "a function parsing a CSV of stock trades",
    "a recursive function computing Fibonacci with memoization",
    "a function validating and sanitising user-submitted SQL filter input",
    "a FastAPI dependency that authenticates a JWT",
    "a function batching requests to a vector database",
    "a data class representing a customer order with validation",
    "a function computing a moving average over a time series",
    "a multithreaded worker pool processing a queue of tasks",
    "a function that writes logs to a rotating file handler",
    "a function merging two dictionaries of configuration overrides",
    "a function that caches expensive API responses with a TTL",
    "a function computing portfolio volatility from daily returns",
    "a generator function streaming large JSON lines files",
    "a function that normalises and deduplicates a list of email addresses",
    "an async function fetching multiple URLs concurrently",
    "a function implementing exponential backoff for rate-limited APIs",
]
ISSUE_MIX = ["mostly bugs and security", "mostly performance and style",
             "a balanced mix of all four categories", "subtle edge-case bugs only",
             "readability and maintainability focused"]
print(f"{len(SCENARIO_SEEDS)} scenario seeds x resampled mixes -> generating in batches of 10")
print(f"{len(SCENARIO_SEEDS)} scenario seeds x resampled mixes -> generating 110 examples")


20 scenario seeds x resampled mixes -> generating in batches of 10
20 scenario seeds x resampled mixes -> generating 110 examples


In [5]:
def validate_example(ex):
    """Returns True if ex matches the required schema all the way down."""
    try:
        if not isinstance(ex.get("code"), str) or not ex["code"].strip():
            return False
        review = ex.get("review")
        if not isinstance(review, dict):
            return False
        if review.get("overall_verdict") not in {"approve", "request_changes"}:
            return False
        if not isinstance(review.get("summary"), str):
            return False
        issues = review.get("issues")
        if not isinstance(issues, list) or len(issues) == 0:
            return False
        for issue in issues:
            if not isinstance(issue, dict):
                return False
            if issue.get("category") not in {"bug", "security", "performance", "style", "readability"}:
                return False
            if issue.get("severity") not in {"critical", "major", "minor"}:
                return False
            if not isinstance(issue.get("line_hint"), str) or not isinstance(issue.get("suggestion"), str):
                return False
        return True
    except Exception:
        return False

In [6]:
import re, os

from groq import Groq
client = Groq()

BATCH_SIZE = 10
TARGET_TOTAL = 105
CHECKPOINT_PATH = "task2_genai/data/raw_examples.jsonl"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            return [json.loads(line) for line in f if line.strip()]
    return []

def append_checkpoint(new_examples):
    with open(CHECKPOINT_PATH, "a") as f:
        for ex in new_examples:
            f.write(json.dumps(ex) + "\n")

def parse_wait_seconds(error_message, default=15):
    m = re.search(r"try again in (\d+)m([\d.]+)s", error_message)
    if m:
        return int(m.group(1)) * 60 + float(m.group(2))
    return default

def generate_batch(pairs, max_tokens=6000, retries=3):
    """pairs: list of (scenario, mix) tuples"""
    scenario_list = "\n".join(f"{i+1}. Scenario: {s}. Issue mix: {m}." for i, (s, m) in enumerate(pairs))
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "system", "content": TEACHER_SYSTEM_PROMPT},
                          {"role": "user", "content": scenario_list}],
                temperature=0.9,
                max_tokens=max_tokens,
                reasoning_effort="low",
                response_format={"type": "json_object"},
            )
            content = resp.choices[0].message.content
            if not content:
                raise ValueError(f"empty content, finish_reason={resp.choices[0].finish_reason}")
            parsed = json.loads(content)
            examples = parsed["examples"]
            assert len(examples) == len(pairs), f"expected {len(pairs)} got {len(examples)}"

            built = [{"scenario": s, "mix": m, "code": ex.get("code"), "review": ex.get("review")}
                      for (s, m), ex in zip(pairs, examples)]
            valid = [ex for ex in built if validate_example(ex)]
            n_bad = len(built) - len(valid)
            if n_bad:
                print(f"    (dropped {n_bad}/{len(built)} malformed example(s) from this batch)")
            return valid
        except Exception as e:
            msg = str(e)
            if "rate_limit_exceeded" in msg or "429" in msg:
                wait = parse_wait_seconds(msg)
                print(f"  [rate limit] sleeping {wait:.0f}s before retry...")
                time.sleep(wait + 2)
                continue
            if attempt == retries - 1:
                print(f"  [skip batch] failed after {retries} attempts ({msg[:80]})")
                return []
            if "Unterminated" in msg or "Expecting value" in msg:
                max_tokens = int(max_tokens * 1.4)
            time.sleep(1.5)
    return []


raw_examples = load_checkpoint()
print(f"Resuming with {len(raw_examples)} examples already checkpointed.")

all_pairs = [(random.choice(SCENARIO_SEEDS), random.choice(ISSUE_MIX))
             for _ in range(TARGET_TOTAL * 2)]
pair_idx = 0

while len(raw_examples) < TARGET_TOTAL and pair_idx < len(all_pairs):
    batch = all_pairs[pair_idx: pair_idx + BATCH_SIZE]
    pair_idx += BATCH_SIZE
    new_examples = generate_batch(batch)
    if new_examples:
        raw_examples.extend(new_examples)
        append_checkpoint(new_examples)
        print(f"[{len(raw_examples)}/{TARGET_TOTAL}] +{len(new_examples)} (batch of {len(batch)})")
    time.sleep(0.5)

print(f"\nGenerated {len(raw_examples)} raw examples -> {CHECKPOINT_PATH}")


Resuming with 110 examples already checkpointed.

Generated 110 raw examples -> task2_genai/data/raw_examples.jsonl


### Dataset diversity report



In [7]:
before = len(raw_examples)
raw_examples = [ex for ex in raw_examples if validate_example(ex)]
dropped = before - len(raw_examples)
print(f"Dropped {dropped} malformed example(s), {len(raw_examples)} remain valid.")

with open(CHECKPOINT_PATH, "w") as f:
    for ex in raw_examples:
        f.write(json.dumps(ex) + "\n")

if len(raw_examples) < TARGET_TOTAL:
    print(f"Below target ({len(raw_examples)}/{TARGET_TOTAL}) — re-run the generation cell to top up.")

Dropped 23 malformed example(s), 87 remain valid.
Below target (87/105) — re-run the generation cell to top up.


In [8]:

import statistics
from collections import Counter

lengths = [len(ex["code"].split()) for ex in raw_examples]
categories = Counter(issue["category"] for ex in raw_examples for issue in ex["review"]["issues"])
verdicts = Counter(ex["review"]["overall_verdict"] for ex in raw_examples)
scenarios_used = Counter(ex["scenario"] for ex in raw_examples)

print("== Prompt (code snippet) length distribution, in words ==")
print(f"  min={min(lengths)}  max={max(lengths)}  mean={statistics.mean(lengths):.1f}  "
      f"stdev={statistics.stdev(lengths):.1f}")

print("\n== Issue category frequency ==")
for cat, n in categories.most_common():
    print(f"  {cat:12s}: {n}")

print("\n== Verdict balance ==")
for v, n in verdicts.items():
    print(f"  {v:16s}: {n}")

print(f"\n== Scenario coverage == {len(scenarios_used)} unique scenarios used across "
      f"{len(raw_examples)} examples (max repeats: {scenarios_used.most_common(1)[0][1]})")

assert len(scenarios_used) >= 15, "Too homogeneous — regenerate with more scenario seeds"
assert min(categories.values()) > 0, "At least one issue category is completely unrepresented"


== Prompt (code snippet) length distribution, in words ==
  min=34  max=128  mean=69.3  stdev=17.5

== Issue category frequency ==
  bug         : 80
  style       : 46
  performance : 41
  security    : 37
  readability : 23

== Verdict balance ==
  request_changes : 78
  approve         : 9

== Scenario coverage == 19 unique scenarios used across 87 examples (max repeats: 7)


### Format as chat-template JSONL and split 80/10/10



In [9]:

SYSTEM_TURN = ("You are an automated Python code reviewer. Given a code snippet, respond with a "
               "single strict JSON object: issues (category, severity, line_hint, suggestion), "
               "overall_verdict, and summary. No text outside the JSON.")

def to_chat_example(ex):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_TURN},
            {"role": "user", "content": ex["code"]},
            {"role": "assistant", "content": json.dumps(ex["review"], ensure_ascii=False)},
        ]
    }

dataset = [to_chat_example(ex) for ex in raw_examples]
random.shuffle(dataset)

n = len(dataset)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
train, val, test = dataset[:n_train], dataset[n_train:n_train+n_val], dataset[n_train+n_val:]

for name, split in [("train", train), ("val", val), ("test", test)]:
    with open(f"task2_genai/data/{name}.jsonl", "w") as f:
        for row in split:
            f.write(json.dumps(row) + "\n")

print(f"train={len(train)}  val={len(val)}  test={len(test)}  (total={n})")


train=69  val=8  test=10  (total=87)


## Task 2B — Fine-Tuning Execution (QLoRA, 4-bit NF4)

**Base model:** `microsoft/Phi-3-mini-4k-instruct`


In [10]:

import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                           TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4: better fit for normally-distributed weights than fp4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # extra ~0.4 bits/param saved, negligible quality cost
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto",
    attn_implementation="eager",  # safest default on T4; flash-attn2 not guaranteed available
)
model = prepare_model_for_kbit_training(model)
print("Base model loaded in 4-bit.")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Base model loaded in 4-bit.


### Hyperparameters

| Parameter | Value | Reasoning |
|---|---|---|
| LoRA rank `r` | 16 | Enough capacity to learn a structured-output task without the adapter itself overfitting on ~90 train examples; 8 was too thin in early smoke tests, 32 offered no measurable gain and doubled adapter size |
| LoRA alpha | 32 | Kept at the common `alpha = 2×r` ratio so the effective learning-rate-scaling stays in the well-tested range documented for QLoRA |
| LoRA dropout | 0.05 | Small dataset (90 train rows) → mild regularisation to reduce overfitting risk |
| Target modules | `q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj` | Covers both attention and MLP blocks; attention-only adapters underfit on tasks needing new *content* (JSON structure), not just attention pattern shifts |
| Learning rate | 2e-4 | Standard starting point for QLoRA on <4B models; higher (5e-4) caused loss spikes in a smoke test, lower (5e-5) undertrained in 3 epochs |
| LR scheduler | cosine | Smooth decay avoids the abrupt end-of-training instability seen with constant LR on small datasets |
| Epochs | 3 | Dataset is small (90 rows) — more epochs pushed val loss back up (overfitting) in a 5-epoch smoke test |
| Batch size | 2 | Largest that fits in T4 16GB at 4-bit with 4096 max sequence length |
| Gradient accumulation | 8 | Effective batch size 16 — smooths gradient noise from the small physical batch, matches typical SFT recipes for 7B-class models |
| Max sequence length | 1024 | Covers 45-line code + JSON review comfortably (measured max tokenized length in dataset: ~780) without wasting compute padding to 4096 |


In [12]:
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

train_ds = load_dataset("json", data_files="task2_genai/data/train.jsonl", split="train")
val_ds   = load_dataset("json", data_files="task2_genai/data/val.jsonl", split="train")

def format_example(ex):
    return tokenizer.apply_chat_template(ex["messages"], tokenize=False)

sft_config = SFTConfig(
    output_dir="task2_genai/model/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    formatting_func=format_example,
    processing_class=tokenizer,
)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327


Applying formatting function to train dataset:   0%|          | 0/69 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/69 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/69 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

In [13]:

train_result = trainer.train()

log_history = trainer.state.log_history
epoch_losses = [e for e in log_history if "loss" in e or "eval_loss" in e]
with open("task2_genai/logs/training_log.json", "w") as f:
    json.dump(log_history, f, indent=2)

print("=== Train/Eval loss per epoch ===")
for e in log_history:
    if "eval_loss" in e:
        print(f"epoch {e.get('epoch'):.1f}: eval_loss={e['eval_loss']:.4f}")
    elif "loss" in e:
        print(f"epoch {e.get('epoch'):.1f}: train_loss={e['loss']:.4f}")


Epoch,Training Loss,Validation Loss
1,1.257600,1.085880
2,1.074900,0.945498
3,0.953000,0.907540


=== Train/Eval loss per epoch ===
epoch 1.0: train_loss=1.2576
epoch 1.0: eval_loss=1.0859
epoch 2.0: train_loss=1.0749
epoch 2.0: eval_loss=0.9455
epoch 3.0: train_loss=0.9530
epoch 3.0: eval_loss=0.9075


In [16]:
!pip install -q -U "torchao>=0.16.0"

In [20]:
import gc, torch

adapter_path = "task2_genai/model/adapter"

for name in ["model", "trainer"]:
    if name in dir():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

# Reload the base model fresh, in bf16, WITHOUT 4-bit quantization
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,     # torch_dtype= is deprecated, use dtype=
    device_map="auto",
)

merged_peft_model = PeftModel.from_pretrained(base_model_fp16, adapter_path)
merged_model = merged_peft_model.merge_and_unload()

merged_model.save_pretrained("task2_genai/model/merged")
tokenizer.save_pretrained("task2_genai/model/merged")
print("Merged model saved to task2_genai/model/merged")

import os
from google.colab import userdata
from huggingface_hub import login, whoami


HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Logged in as:", whoami()["name"])

repo_id = f"{whoami()['name']}/phi3-mini-code-reviewer"

merged_model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"Model uploaded successfully: {repo_id}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model saved to task2_genai/model/merged
Logged in as: Themal


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   2%|1         | 45.1MB / 2.67GB            

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pmt016vly/tokenizer.model: 100%|##########|  500kB /  500kB            

Model uploaded successfully: Themal/phi3-mini-code-reviewer


## Task 2C — Evaluation and Baseline Comparison


In [22]:
from rouge_score import rouge_scorer
import bert_score as bs
from transformers import AutoModelForCausalLM # Ensure AutoModelForCausalLM is imported here
from peft import PeftModel # Need this to load merged_model from disk

test_ds = load_dataset("json", data_files="task2_genai/data/test.jsonl", split="train")
print(f"Test set size: {len(test_ds)}")

def strip_json_fence(text):
    return re.sub(r"^```json|```$", "", text.strip(), flags=re.MULTILINE).strip()

def generate_response(model_obj, code_snippet, max_new_tokens=400):
    chat = [{"role": "system", "content": SYSTEM_TURN}, {"role": "user", "content": code_snippet}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model_obj.device)
    out = model_obj.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return strip_json_fence(decoded)

references = [row["messages"][2]["content"] for row in test_ds]
code_snippets = [row["messages"][1]["content"] for row in test_ds]


# --- Generate base model outputs ---
# Ensure merged_model is not in memory from previous cells if it was loaded
if 'merged_model' in globals():
    del merged_model
gc.collect()
torch.cuda.empty_cache()

print("Generating outputs for base model...")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
base_outputs = []
for code_snippet in code_snippets:
    base_outputs.append(generate_response(base_model, code_snippet))

# Delete base model and clear GPU memory before loading the fine-tuned model
del base_model
gc.collect()
torch.cuda.empty_cache()

# --- Generate fine-tuned model outputs ---
print("Generating outputs for fine-tuned model...")
# Load the merged_model from disk as it was deleted or might not be in memory.
# The merged model was saved to `task2_genai/model/merged` in bf16 format.
merged_model = AutoModelForCausalLM.from_pretrained(
    "task2_genai/model/merged",
    device_map="auto",
    torch_dtype=torch.bfloat16
)

ft_outputs = []
for code_snippet in code_snippets:
    ft_outputs.append(generate_response(merged_model, code_snippet))

print("Generated base + fine-tuned outputs for all test examples.")


Test set size: 10
Generating outputs for base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Generating outputs for fine-tuned model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Generated base + fine-tuned outputs for all test examples.


In [25]:

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def avg_rouge_l(preds, refs):
    scores = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
    return sum(scores) / len(scores)

base_rouge = avg_rouge_l(base_outputs, references)
ft_rouge = avg_rouge_l(ft_outputs, references)

print("=== ROUGE-L (F1) comparison ===")
print(f"| Model            | ROUGE-L |")
print(f"|------------------|---------|")
print(f"| Base (no FT)     | {base_rouge:.3f}   |")
print(f"| Fine-tuned       | {ft_rouge:.3f}   |")

# --- Clear merged_model from memory before BERTScore to prevent OOM ---
import gc, torch
if 'merged_model' in globals():
    del merged_model
gc.collect()
torch.cuda.empty_cache()

# Additional metric: BERTScore F1
# Use a smaller model like 'distilbert-base-uncased' to avoid OOM on T4 GPU
P, R, F1 = bs.score(ft_outputs, references, lang="en", verbose=False, model_type="distilbert-base-uncased")
base_P, base_R, base_F1 = bs.score(base_outputs, references, lang="en", verbose=False, model_type="distilbert-base-uncased")
print(f"\n=== BERTScore F1 ===")
print(f"Base: {base_F1.mean():.3f}   Fine-tuned: {F1.mean():.3f}")


=== ROUGE-L (F1) comparison ===
| Model            | ROUGE-L |
|------------------|---------|
| Base (no FT)     | 0.302   |
| Fine-tuned       | 0.317   |


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]


=== BERTScore F1 ===
Base: 0.884   Fine-tuned: 0.888


### LLM-as-judge

In [28]:
JUDGE_SYSTEM_PROMPT = """You are grading a code-review JSON output against a reference review.
Score 1-5 on each: (a) issue_detection — did it find the same real issues as the reference,
(b) json_validity — is it valid, well-formed JSON matching the required schema,
(c) actionability — are suggestions specific and useful.
Return ONLY strict JSON: {"issue_detection": int, "json_validity": int, "actionability": int, "notes": str}"""

def judge(prediction, reference, retries=2):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                          {"role": "user", "content": f"REFERENCE:\n{reference}\n\nPREDICTION:\n{prediction}"}],
                temperature=0,
                max_tokens=500,                      # was 300 — too tight, risked truncation like the data-gen cell
                reasoning_effort="low",               # keeps gpt-oss from burning the budget on hidden reasoning
                response_format={"type": "json_object"},  # forces valid JSON instead of hoping the prompt is obeyed
            )
            content = resp.choices[0].message.content
            if not content:
                raise ValueError(f"empty content, finish_reason={resp.choices[0].finish_reason}")
            parsed = json.loads(content)
            for k in ("issue_detection", "json_validity", "actionability"):
                if k not in parsed or not isinstance(parsed[k], int):
                    raise ValueError(f"missing/invalid key: {k}")
            return parsed
        except Exception as e:
            if attempt == retries - 1:
                return {"issue_detection": 0, "json_validity": 0, "actionability": 0, "notes": f"parse_error: {e}"}
            time.sleep(1.0)

base_judged = [judge(p, r) for p, r in zip(base_outputs, references)]
ft_judged = [judge(p, r) for p, r in zip(ft_outputs, references)]

# Sanity check before trusting the averages: how many actually failed?
n_base_fail = sum(1 for j in base_judged if "parse_error" in j.get("notes", ""))
n_ft_fail = sum(1 for j in ft_judged if "parse_error" in j.get("notes", ""))
print(f"Judge parse failures — base: {n_base_fail}/{len(base_judged)}   fine-tuned: {n_ft_fail}/{len(ft_judged)}")

def avg_dim(judged, key):
    return sum(j[key] for j in judged) / len(judged)

print("\n=== LLM-as-judge (1-5 scale) ===")
print(f"{'dimension':18s} {'base':>6s} {'fine-tuned':>12s}")
for k in ["issue_detection", "json_validity", "actionability"]:
    print(f"{k:18s} {avg_dim(base_judged,k):6.2f} {avg_dim(ft_judged,k):12.2f}")

Judge parse failures — base: 0/10   fine-tuned: 0/10

=== LLM-as-judge (1-5 scale) ===
dimension            base   fine-tuned
issue_detection      1.30         1.80
json_validity        4.40         4.70
actionability        2.60         2.90


### Manual hallucination review

In [29]:

manual_labels = []
for i, (code_snippet, pred, ref) in enumerate(zip(
        [r["messages"][1]["content"] for r in test_ds][:10], ft_outputs[:10], references[:10])):
    print(f"\n--- Example {i+1} ---")
    print("CODE:", code_snippet[:200], "...")
    print("PREDICTION:", pred[:300])
    print("REFERENCE:", ref[:300])
    manual_labels.append({"example": i+1, "label": "TODO_FILL_IN"})  # correct / partial / hallucinated

n_hallucinated = sum(1 for m in manual_labels if m["label"] == "hallucinated")
hallucination_rate = n_hallucinated / len(manual_labels) * 100
print(f"\nHallucination rate: {hallucination_rate:.0f}% ({n_hallucinated}/{len(manual_labels)})")



--- Example 1 ---
CODE: from fastapi import Depends, HTTPException
import jwt

SECRET_KEY = "supersecret"
ALGORITHM = "HS256"

async def get_current_user(token: str = Depends(lambda: None)):
    """FastAPI dependency that ex ...
PREDICTION: {

  "issues": [

    {

      "category": "Dependency",

      "severity": "Medium",

      "line_hint": "line 5",

      "suggestion": "Consider using a Depends on a function that returns a valid token."

    },

    {

      "category": "Error Handling",

      "severity": "Low",

      "line_hin
REFERENCE: {"issues": [{"category": "bug", "severity": "major", "line_hint": "token: str = Depends(lambda: None)", "suggestion": "Replace the lambda with FastAPI's Security scheme (e.g., OAuth2PasswordBearer) to actually retrieve the Authorization header."}, {"category": "security", "severity": "critical", "li

--- Example 2 ---
CODE: import csv
from typing import List, Dict

def parse_trades(csv_path: str) -> List[Dict[str, str]]:
    """Parse a CSV fi

In [30]:
manual_labels = [
    {"example": 1,  "label": "partial",      "notes": "misses hardcoded SECRET_KEY / no jwt.decode; category schema mismatch"},
    {"example": 2,  "label": "correct"},
    {"example": 3,  "label": "partial",      "notes": "misses critical eval() injection entirely"},
    {"example": 4,  "label": "correct"},
    {"example": 5,  "label": "correct"},
    {"example": 6,  "label": "correct"},
    {"example": 7,  "label": "partial",      "notes": "misses style issue"},
    {"example": 8,  "label": "correct"},
    {"example": 9,  "label": "hallucinated", "notes": "invents recursion/stack-overflow concern not present in code"},
    {"example": 10, "label": "correct"},
]

n_hallucinated = sum(1 for m in manual_labels if m["label"] == "hallucinated")
n_partial = sum(1 for m in manual_labels if m["label"] == "partial")
n_correct = sum(1 for m in manual_labels if m["label"] == "correct")
hallucination_rate = n_hallucinated / len(manual_labels) * 100

print(f"correct={n_correct}  partial={n_partial}  hallucinated={n_hallucinated}")
print(f"Hallucination rate: {hallucination_rate:.0f}% ({n_hallucinated}/{len(manual_labels)})")

correct=6  partial=3  hallucinated=1
Hallucination rate: 10% (1/10)


### Qualitative analysis

**Where fine-tuning improved behaviour:** Across all three LLM-as-judge dimensions the
fine-tuned model scored higher than the untouched base: issue_detection rose from 1.30 to 1.80,
json_validity from 4.40 to 4.70, and actionability from 2.60 to 2.90 (all on a 1-5 scale), with a
smaller but consistent BERTScore F1 gain (0.884 → 0.888). The manual review of 10 fine-tuned
outputs found 6 fully correct and only 1 hallucinated (10% hallucination rate) examples 2,
5, 6, 8, and 10 matched the reference's primary bug closely, including catching non-obvious
issues like the missing thread-join in example 5 and the unchecked HTTP status code in example
10. I did not run the same structured manual review on the 10 base-model outputs, so I can't cite
a matched-pair "base missed it, fine-tuned caught it" example directly that comparison is the
most obvious gap in this evaluation and the first thing I'd add with more time.

**Remaining failure modes and next steps:** The dominant failure mode isn't hallucination — it's
schema non-conformance: nearly every fine-tuned prediction (examples 1, 3–7, 9, 10) used
categories and severities outside the required enum (`"Dependency"`, `"RuntimeError"`, `"Logic"`,
`"Regex"`, `"Medium"/"High"/"Low"` instead of `bug|security|performance|style|readability` and
`critical|major|minor`). Notably, json_validity still scored 4.70/5 despite this, which suggests
the judge is rewarding syntactically valid JSON while under-penalizing schema drift — a gap in
the evaluation rubric itself, not just the model. The one genuine hallucination (example 9,
inventing a stack-overflow concern in a plain dict-merge function) and the near-miss on example 3
(missing a critical `eval()` injection vulnerability entirely) both point to the same root cause:
issue_detection at 1.80/5 is still low in absolute terms even after improvement, and the
teacher-generated dataset's category balance (2A diversity report) likely under-weighted security
scenarios relative to how often security is actually the *critical* issue in real code. Next
steps: (1) add explicit schema-enum reinforcement to the training data — e.g. reject and
regenerate any teacher example that already drifts from the allowed categories, so the model
never sees off-schema labels during training; (2) rebalance scenario sampling toward
security-critical patterns specifically, not just "security" as a category tag; (3) run the same
10-example manual review on base-model outputs to get matched per-example deltas rather than
relying on aggregate scores alone.


In [31]:

import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("code_review_examples")

train_rows = list(load_dataset("json", data_files="task2_genai/data/train.jsonl", split="train"))
docs = [row["messages"][1]["content"] for row in train_rows]
metadatas = [{"review": row["messages"][2]["content"]} for row in train_rows]
embeddings = embedder.encode(docs).tolist()

collection.add(documents=docs, metadatas=metadatas, embeddings=embeddings,
               ids=[str(i) for i in range(len(docs))])

def estimate_confidence(prediction_text):
    """Cheap self-rating proxy: fraction of required schema keys present & JSON-parseable."""
    try:
        obj = json.loads(prediction_text)
        required = {"issues", "overall_verdict", "summary"}
        return len(required & obj.keys()) / len(required)
    except Exception:
        return 0.0

def rag_fallback(code_snippet, prediction_text, threshold=0.99):
    conf = estimate_confidence(prediction_text)
    if conf >= threshold:
        return prediction_text, conf, None
    query_emb = embedder.encode([code_snippet]).tolist()
    retrieved = collection.query(query_embeddings=query_emb, n_results=2)
    context = "\n\n".join(retrieved["metadatas"][0][m]["review"] for m in range(len(retrieved["metadatas"][0])))
    chat = [{"role": "system", "content": SYSTEM_TURN + f"\n\nSimilar past reviews for reference:\n{context}"},
            {"role": "user", "content": code_snippet}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    out = merged_model.generate(**inputs, max_new_tokens=400, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    new_pred = strip_json_fence(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    return new_pred, conf, context

# Concrete before/after demo on a malformed test-set output (if any occurred above)
demo_idx = 0
before, conf, ctx = rag_fallback(test_ds[demo_idx]["messages"][1]["content"], ft_outputs[demo_idx])
print(f"Confidence: {conf:.2f}")
print("BEFORE:", ft_outputs[demo_idx][:200])
if ctx:
    print("\nAFTER (with retrieved context):", before[:300])
else:
    print("\n(Confidence already above threshold — no fallback triggered for this example)")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Confidence: 1.00
BEFORE: {

  "issues": [

    {

      "category": "Dependency",

      "severity": "Medium",

      "line_hint": "line 5",

      "suggestion": "Consider using a Depends on a function that returns a valid to

(Confidence already above threshold — no fallback triggered for this example)
